# EmotionLens · Knowledge Distillation

**Goal**: distill HSEmotion (EfficientNet-B2, AffectNet, 260px) into a lightweight **ResNet18 student** (224px, 7-class).

Output = `best_distilled.pt` that drop-in replaces `final_outputs/best.pt` in the EmotionLens project.

**Runtime**: Colab free T4 GPU, ~60–90 min for 15 epochs.

**Before running**: `Runtime → Change runtime type → GPU (T4)`.

## 0. Environment check + install

In [ ]:
!nvidia-smi | head -15
# Pin datasets < 3.0 — HF removed dataset-script support in 3.x, but the
# public FER2013 mirror we use is still script-based. 2.x also plays nicer
# with older HSEmotion / timm pins.
!pip install -q 'timm==0.9.16' hsemotion 'datasets<3.0'

In [ ]:
import os, io, math, time, urllib.request, warnings
warnings.filterwarnings('ignore')

import torch

# HSEmotion .pt was saved on CUDA and is a FULL pickled model object.
# Two defaults to override in torch.load:
#   - map_location='cpu'  → works on CPU-only runtimes
#   - weights_only=False  → PyTorch 2.6+ made this True by default, which
#     refuses to unpickle model classes. We know the source is trusted.
# Guard is idempotent: re-running this cell won't wrap torch.load again.
if not getattr(torch.load, '_emolens_patched', False):
    _true_torch_load = torch.load
    def _safe_load(*a, **kw):
        kw.setdefault('map_location', 'cpu')
        kw.setdefault('weights_only', False)
        return _true_torch_load(*a, **kw)
    _safe_load._emolens_patched = True
    torch.load = _safe_load

import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('  GPU:', torch.cuda.get_device_name(0))
    print('  Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('torch.load patched:', getattr(torch.load, '_emolens_patched', False))

## 1. Load HSEmotion teacher

Downloads `enet_b2_7.pt` (EfficientNet-B2, 7-class AffectNet). Frozen during training.

In [ ]:
os.makedirs(os.path.expanduser('~/.hsemotion'), exist_ok=True)
TEACHER_PATH = os.path.expanduser('~/.hsemotion/enet_b2_7.pt')
if not os.path.exists(TEACHER_PATH):
    url = 'https://raw.githubusercontent.com/sb-ai-lab/EmotiEffLib/main/models/affectnet_emotions/enet_b2_7.pt'
    print('Downloading teacher …')
    urllib.request.urlretrieve(url, TEACHER_PATH)
print('Teacher size:', round(os.path.getsize(TEACHER_PATH)/1e6, 1), 'MB')

from hsemotion.facial_emotions import HSEmotionRecognizer
_rec = HSEmotionRecognizer(model_name='enet_b2_7', device=DEVICE)
teacher = _rec.model.to(DEVICE).eval()
for p in teacher.parameters():
    p.requires_grad = False
print('Teacher params:', round(sum(p.numel() for p in teacher.parameters())/1e6, 2), 'M')

OUR_LABELS = ['neutral', 'happiness', 'surprise', 'sadness', 'anger', 'disgust', 'fear']
HSE_LABELS = ['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise']
HSE_TO_OURS = torch.tensor([HSE_LABELS.index(e) for e in OUR_LABELS], device=DEVICE)
print('HSE_TO_OURS remap:', HSE_TO_OURS.tolist())

## 2. Load FER2013 dataset

Public 32K face images with 7-class labels. No license needed.

If the primary source fails, swap `DS_NAME` for another HF mirror (e.g. `dpdl-benchmark/fer2013`).

In [ ]:
from datasets import load_dataset

DS_NAME = 'Jeneral/fer-2013'
raw = load_dataset(DS_NAME)
print(raw)

train_hf = raw['train']
val_key = 'test' if 'test' in raw else 'validation'
val_hf = raw[val_key]
print(f'Train: {len(train_hf)} | Val: {len(val_hf)}')

# This mirror stores fields as 'img_bytes' + 'labels'. The label field is a
# plain int (not a ClassLabel), so we hard-code the canonical FER2013 order.
FER_LABELS = ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise', 'neutral']
print('FER label order (canonical):', FER_LABELS)

OUR_OF_FER = [OUR_LABELS.index(e) for e in FER_LABELS]
print('For FER idx i, our idx =', OUR_OF_FER)

sample = train_hf[0]
print('\nSample keys:', list(sample.keys()))
print('img_bytes type:', type(sample['img_bytes']))
print('label:', sample['labels'])

## 3. Transforms + DataLoaders

Student sees 224 crops with augmentation; teacher sees 260 crops of the same image (deterministic).

In [ ]:
import io
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf_student = transforms.Compose([
    transforms.Lambda(lambda im: im.convert('RGB') if im.mode != 'RGB' else im),
    transforms.Resize((240, 240)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.15, 0.15, 0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf_student = transforms.Compose([
    transforms.Lambda(lambda im: im.convert('RGB') if im.mode != 'RGB' else im),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

tf_teacher = transforms.Compose([
    transforms.Lambda(lambda im: im.convert('RGB') if im.mode != 'RGB' else im),
    transforms.Resize((260, 260)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def _to_pil(x):
    """Decode img_bytes to a PIL image. Handles raw bytes, HF's dict wrapper,
    or an already-PIL image transparently."""
    if isinstance(x, Image.Image):
        return x
    if isinstance(x, dict) and 'bytes' in x and x['bytes'] is not None:
        return Image.open(io.BytesIO(x['bytes']))
    if isinstance(x, (bytes, bytearray)):
        return Image.open(io.BytesIO(x))
    raise TypeError(f'Unsupported image field type: {type(x)}')

class DistillDataset(Dataset):
    def __init__(self, hf_split, student_tf, teacher_tf):
        self.hf = hf_split
        self.s_tf = student_tf
        self.t_tf = teacher_tf
    def __len__(self):
        return len(self.hf)
    def __getitem__(self, i):
        row = self.hf[i]
        pil = _to_pil(row['img_bytes'])
        s_img = self.s_tf(pil)
        t_img = self.t_tf(pil)
        y_ours = OUR_OF_FER[row['labels']]
        return s_img, t_img, y_ours

NUM_WORKERS = 2
BATCH = 64

train_ds = DistillDataset(train_hf, train_tf_student, tf_teacher)
val_ds   = DistillDataset(val_hf,   eval_tf_student,  tf_teacher)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print('Train batches:', len(train_loader), '| Val batches:', len(val_loader))

# Sanity — decode one sample end-to-end
s, t, y = train_ds[0]
print(f'\nSample tensors: student={tuple(s.shape)} teacher={tuple(t.shape)} label={y} ({OUR_LABELS[y]})')

## 4. Build ResNet18 student

ImageNet-pretrained backbone + 7-class head. Output indices match `OUR_LABELS` order (same as `best.pt`).

In [ ]:
# Optional: mount Google Drive to initialise the student from best.pt.
# Starting from best.pt (already knows FER) rather than fresh ImageNet weights
# gives 2-4 pp higher final accuracy for the same epoch budget.
BEST_PT_PATH = '/content/drive/MyDrive/best.pt'
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except Exception as _e:
    print('Drive mount skipped:', _e)

student = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
student.fc = nn.Linear(student.fc.in_features, 7)

if os.path.exists(BEST_PT_PATH):
    _ck = torch.load(BEST_PT_PATH, map_location='cpu', weights_only=False)
    _sd = _ck.get('model_state', _ck)
    missing, unexpected = student.load_state_dict(_sd, strict=False)
    print(f'Student init from best.pt ({BEST_PT_PATH})')
    if missing:    print(f'  missing keys:    {len(missing)}')
    if unexpected: print(f'  unexpected keys: {len(unexpected)}')
    _val_acc_best = _ck.get('val_acc', None)
    if _val_acc_best is not None:
        print(f'  best.pt self-reported val_acc = {_val_acc_best*100:.2f}%')
else:
    print(f'Student init from ImageNet weights (best.pt not found at {BEST_PT_PATH})')

student = student.to(DEVICE)
print(f'\nStudent params: {sum(p.numel() for p in student.parameters())/1e6:.2f} M')
print(f'Teacher params: {sum(p.numel() for p in teacher.parameters())/1e6:.2f} M')

## 5. Distillation loss + training

Hinton et al. 2015 combined loss:
$$L = \alpha \cdot T^2 \cdot KL(\sigma(z_s / T) \parallel \sigma(z_t / T)) + (1-\alpha) \cdot CE(z_s, y)$$

- **T = 4** (softens both distributions during transfer)
- **α = 0.7** (heavy on distillation, light on hard labels)
- Cosine LR schedule, AdamW, 15 epochs

In [ ]:
EPOCHS = 15
LR = 1e-4
WD = 1e-4
TEMPERATURE = 4.0
ALPHA = 0.7

optim_ = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WD)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim_, T_max=EPOCHS * len(train_loader))
scaler = torch.cuda.amp.GradScaler() if DEVICE == 'cuda' else None

def distill_loss(z_student, z_teacher_our_order, y):
    T = TEMPERATURE
    log_p_s = F.log_softmax(z_student / T, dim=1)
    p_t     = F.softmax(z_teacher_our_order / T, dim=1)
    kd = F.kl_div(log_p_s, p_t, reduction='batchmean') * (T * T)
    ce = F.cross_entropy(z_student, y)
    return ALPHA * kd + (1 - ALPHA) * ce, kd.item(), ce.item()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    hit, total = 0, 0
    for s_img, _, y in loader:
        s_img, y = s_img.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits = model(s_img)
        pred = logits.argmax(1)
        hit += (pred == y).sum().item()
        total += y.size(0)
    return hit / max(1, total)

history = []
best_val = 0.0
print(f'Starting distillation: T={TEMPERATURE}, alpha={ALPHA}, epochs={EPOCHS}')

for ep in range(1, EPOCHS + 1):
    student.train()
    t0 = time.time()
    losses, kds, ces = [], [], []
    for s_img, t_img, y in train_loader:
        s_img = s_img.to(DEVICE, non_blocking=True)
        t_img = t_img.to(DEVICE, non_blocking=True)
        y     = y.to(DEVICE, non_blocking=True)

        with torch.no_grad():
            z_t_hse = teacher(t_img)
            z_t_ours = z_t_hse.index_select(1, HSE_TO_OURS)

        optim_.zero_grad(set_to_none=True)
        if scaler is not None:
            with torch.cuda.amp.autocast():
                z_s = student(s_img)
                loss, kd_v, ce_v = distill_loss(z_s, z_t_ours, y)
            scaler.scale(loss).backward()
            scaler.step(optim_)
            scaler.update()
        else:
            z_s = student(s_img)
            loss, kd_v, ce_v = distill_loss(z_s, z_t_ours, y)
            loss.backward()
            optim_.step()
        sched.step()

        losses.append(loss.item()); kds.append(kd_v); ces.append(ce_v)

    val_acc = evaluate(student, val_loader)
    dt = time.time() - t0
    row = {
        'epoch': ep,
        'train_loss': float(np.mean(losses)),
        'kd_loss':    float(np.mean(kds)),
        'ce_loss':    float(np.mean(ces)),
        'val_acc':    val_acc,
        'lr':         sched.get_last_lr()[0],
        'sec':        dt,
    }
    history.append(row)
    print(f"epoch {ep:2d}/{EPOCHS} | loss={row['train_loss']:.3f} (kd={row['kd_loss']:.3f} ce={row['ce_loss']:.3f}) | val_acc={val_acc*100:.2f}% | lr={row['lr']:.2e} | {dt:.0f}s")
    if val_acc > best_val:
        best_val = val_acc
        ckpt = {
            'model_state':    student.state_dict(),
            'classes':        OUR_LABELS,
            'img_size':       224,
            'epoch':          ep,
            'val_acc':        val_acc,
            'distilled_from': 'HSEmotion enet_b2_7',
            'distill_config': {
                'T': TEMPERATURE, 'alpha': ALPHA,
                'lr': LR, 'wd': WD, 'batch': BATCH, 'epochs': EPOCHS,
                'dataset': DS_NAME,
            },
        }
        torch.save(ckpt, 'best_distilled.pt')
        print(f'  saved best_distilled.pt (val_acc={val_acc*100:.2f}%)')

print(f'\nBest val_acc: {best_val*100:.2f}%')

## 6. Compare student vs teacher on val set

Sanity check: student should retain 90%+ of teacher's accuracy.

In [ ]:
@torch.no_grad()
def evaluate_teacher(loader):
    hit, total = 0, 0
    for _, t_img, y in loader:
        t_img, y = t_img.to(DEVICE), y.to(DEVICE)
        z = teacher(t_img).index_select(1, HSE_TO_OURS)
        pred = z.argmax(1)
        hit += (pred == y).sum().item()
        total += y.size(0)
    return hit / max(1, total)

# Reload best distilled student
ckpt = torch.load('best_distilled.pt', map_location=DEVICE)
student.load_state_dict(ckpt['model_state'])

t_acc = evaluate_teacher(val_loader)
s_acc = evaluate(student, val_loader)

# Also evaluate the raw best.pt for a full 3-way comparison
b_acc = None
if os.path.exists(BEST_PT_PATH):
    _best = tvm.resnet18(weights=None)
    _best.fc = nn.Linear(_best.fc.in_features, 7)
    _ck = torch.load(BEST_PT_PATH, map_location='cpu', weights_only=False)
    _best.load_state_dict(_ck.get('model_state', _ck), strict=False)
    _best = _best.to(DEVICE).eval()
    b_acc = evaluate(_best, val_loader)

print(f'Teacher   (HSEmotion)         val_acc = {t_acc*100:.2f}%')
if b_acc is not None:
    print(f'Baseline  (best.pt raw)       val_acc = {b_acc*100:.2f}%')
print(f'Student   (distilled ResNet18) val_acc = {s_acc*100:.2f}%   ({s_acc/max(1e-6,t_acc)*100:.1f}% of teacher)')
if b_acc is not None:
    print(f'\nDistilled uplift over best.pt: {(s_acc - b_acc)*100:+.2f} pp')

from collections import Counter
hits = Counter(); totals = Counter()
student.eval()
with torch.no_grad():
    for s_img, _, y in val_loader:
        s_img, y = s_img.to(DEVICE), y.to(DEVICE)
        pred = student(s_img).argmax(1)
        for pi, yi in zip(pred.tolist(), y.tolist()):
            totals[yi] += 1
            if pi == yi:
                hits[yi] += 1
print('\nStudent per-class val accuracy:')
for i, name in enumerate(OUR_LABELS):
    if totals[i] == 0: continue
    print(f'  {name:10s}: {hits[i]/totals[i]*100:5.1f}%  (n={totals[i]})')

## 7. Download the distilled checkpoint

In [ ]:
from google.colab import files
print('Downloading best_distilled.pt …')
files.download('best_distilled.pt')

## 8. Deploy in EmotionLens

1. Copy `best_distilled.pt` into `emotion-app/final_outputs/`.
2. Edit `backend/config.py` — replace the `general` entry:

```python
'general': _resnet18_entry('best_distilled.pt', 'General · Distilled ResNet18'),
```

3. Restart `python main.py`. The general dropdown now runs your distilled ResNet18 — same architecture as your original best.pt, but with HSEmotion's AffectNet-scale knowledge baked in.

**Paper narrative to add**:

> To combine SOTA accuracy with lightweight inference, we distill HSEmotion (EfficientNet-B2, AffectNet, 260px) into ResNet18 (224px, 11M params) using FER2013 with soft-label supervision (T=4, alpha=0.7). The distilled generic retains X% of teacher accuracy at 2× inference speed. Our personalization pipeline fine-tunes each user's model from this stronger generic backbone.

**Optional**: re-run `finetune.py` for each user against `best_distilled.pt` instead of `best.pt` — the resulting personalized models will inherit the stronger backbone.